In [5]:
import sys, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query

### Pulling a single Data Set

In [8]:
london = run_query("SELECT * FROM week06.london_2014")
london.head()

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees<br />
0,2014-1-1,11,8,6,9,7,4,94,86,73,...,31,11,2,40,26,66.0,9.91,4.0,Rain,186<br />
1,2014-1-2,11,9,6,9,6,4,94,81,60,...,31,15,5,32,21,50.0,1.02,2.0,Rain,214<br />
2,2014-1-3,11,8,6,9,5,2,94,76,54,...,31,12,0,50,29,69.0,7.11,2.0,Rain-Thunderstorm,219<br />
3,2014-1-4,9,6,3,8,6,2,93,85,65,...,31,12,5,35,21,39.0,9.91,4.0,Rain,211<br />
4,2014-1-5,12,7,1,11,4,-1,100,88,77,...,31,14,5,35,16,NaN,0.25,5.0,Rain,199<br />


## Combining the Six Cities

In [9]:

cities = run_query("""
    SELECT 'London' AS city, GMT AS observation_date,
           [Mean TemperatureC]
    FROM week06.london_2014

    UNION ALL

    SELECT 'Beijing', [Date], [Mean TemperatureC]
    FROM week06.beijing_pek_2014

    UNION ALL

    SELECT 'Brasilia', [Date], [Mean TemperatureC]
    FROM week06.brasilia_bsb_2014

    UNION ALL

    SELECT 'CapeTown', [Date], [Mean TemperatureC]
    FROM week06.capetown_cpt_2014

    UNION ALL

    SELECT 'Delhi', [Date], [Mean TemperatureC]
    FROM week06.delhi_del_2014

    UNION ALL

    SELECT 'Moscow', [Date], [Mean TemperatureC]
    FROM week06.moscow_svo_2014
""")

print(cities.shape)
cities.head()


(2190, 3)


,city,observation_date,Mean TemperatureC
0,London,2014-1-1,8
1,London,2014-1-2,9
2,London,2014-1-3,8
3,London,2014-1-4,6
4,London,2014-1-5,7


## Aggregate
Average temperature per city, computed in SQ

In [14]:
avg_by_city = run_query("""
    SELECT 'London' AS city, AVG([Mean TemperatureC]) AS avg_temp FROM week06.london_2014
    UNION ALL SELECT 'Delhi', AVG([Mean TemperatureC]) FROM week06.delhi_del_2014
""")

avg_by_city.head()

,city,avg_temp
0,London,12
1,Delhi,13


pull raw rows and aggregate in pandas:

In [15]:
cities.groupby("city")["Mean TemperatureC"].mean()

city
Beijing     13.364384
Brasilia    22.895890
CapeTown    17.567123
Delhi       13.709589
London      12.328767
Moscow       5.994521
Name: Mean TemperatureC, dtype: float64

Save to CSV

In [16]:
cities.to_csv("cities_mean_temp.csv", index=False)

WHO population & TB data

In [18]:

who = run_query("SELECT * FROM week06.who_pop_tb_all")
who.head()

,Country,Population (1000s),TB deaths
0,Afghanistan,30552,13000.00
1,Albania,3173,20.00
2,Algeria,39208,5100.00
3,Andorra,79,0.26
4,Angola,21472,6900.00
